<a href="https://colab.research.google.com/github/elithecreator06/neural-networks-cnn-classification/blob/main/neural-network-and-cifar10-cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
#Part 1
#Load Iris dataset
iris = datasets.load_iris()
x = iris.data
y = iris.target.reshape(-1,1)

#One-hot encode the labels
encoder = OneHotEncoder(sparse_output = False)
y = encoder.fit_transform(y)

#Split data (70% train, 30% test)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.3, random_state = 42
)

#Scale the features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

#Nerual Network architecture
input_size = 4
hidden1 = 2
hidden2 = 2
output_size = 3

learning_rate = 0.05
epochs = 5000

In [ ]:
#Initialise weights
np.random.seed(42)

W1 = np.random.randn(input_size, hidden1)
b1 = np.zeros((1, hidden1))

W2 = np.random.randn(hidden1, hidden2)
b2 = np.zeros((1, hidden2))

W3 = np.random.randn(hidden2, output_size)
b3 = np.zeros((1, output_size))

#Activation functions
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
  return x * (1 - x)

def softmax(x):
  exp = np.exp(x - np.max(x, axis = 1, keepdims = True))
  return exp / np.sum(exp, axis = 1, keepdims = True)

In [ ]:
#Training
for epoch in range(epochs):

  #Forward pass
  z1 = np.dot(x_train, W1) + b1
  a1 = sigmoid(z1)

  z2 = np.dot(a1, W2) + b2
  a2 = sigmoid(z2)

  z3 = np.dot(a2, W3) + b3
  output = softmax(z3)

  #Backpropagation
  error = output - y_train

  dW3 = np.dot(a2.T, error)
  db3 = np.sum(error, axis = 0, keepdims = True)

  error2 = np.dot(error, W3.T) * sigmoid_derivative(a2)

  dW2 = np.dot(a1.T, error2)
  db2 = np.sum(error2, axis = 0, keepdims = True)

  error1 = np.dot(error2, W2.T) * sigmoid_derivative(a1)

  dW1 = np.dot(x_train.T, error1)
  db1 = np.sum(error1, axis = 0, keepdims = True)

  #Update weights
  W3 -= learning_rate * dW3
  b3 -= learning_rate * db3

  W2 -= learning_rate * dW2
  b2 -= learning_rate * db2

  W1 -= learning_rate * dW1
  b1 -= learning_rate * db1


#Testing
z1 = np.dot(x_test, W1) + b1
a1 = sigmoid(z1)

z2 = np.dot(a1, W2) + b2
a2 = sigmoid(z2)

z3 = np.dot(a2, W3) + b3
predictions = softmax(z3)

predicted_classes = np.argmax(predictions, axis = 1)
actual_classes = np.argmax(y_test, axis = 1)

accuracy = np.mean(predicted_classes == actual_classes)

print("Classification Accuracy: ", accuracy)

Classification Accuracy:  1.0


In [ ]:
#Part 2
from keras.datasets import cifar10
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
#Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

#Combine original training and testing data
x = np.concatenate((x_train, x_test), axis = 0)
y = np.concatenate((y_train, y_test), axis = 0)

#Normalise images
x = x.astype("float32") / 255.0

#Split into 70% training and 30% testing
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.30,
    random_state = 42,
    stratify = y
)

In [ ]:
#Save original labels for evaluatio
y_test_labels = y_test

#One-hot encode labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

#Build CNN model
model = Sequential()

model.add(Conv2D(32, (3,3), activation = 'relu', input_shape = (32,32,3)))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64, (3,3), activation = 'relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())

model.add(Dense(128, activation = 'relu'))
model.add(Dense(10, activation = 'softmax'))

#Compile model
model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#Train model
model.fit(
    x_train,
    y_train,
    epochs = 10,
    batch_size = 64,
    validation_split = 0.1,
    verbose = 1
)

#Predict
predictions = model.predict(x_test)

predicted_labels = np.argmax(predictions, axis = 1)
true_labels = y_test_labels.flatten()

#Accuracy
accuracy = accuracy_score(true_labels, predicted_labels)

#F1 Score
f1 = f1_score(true_labels, predicted_labels, average = 'weighted')

print("Classification Accuracy: ", accuracy)
print("F1 Score: ", f1)

Epoch 1/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 56s 91ms/step - accuracy: 0.4326 - loss: 1.5692 - val_accuracy: 0.5333 - val_loss: 1.3185
Epoch 2/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 54s 91ms/step - accuracy: 0.5738 - loss: 1.2059 - val_accuracy: 0.6121 - val_loss: 1.1222
Epoch 3/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 54s 91ms/step - accuracy: 0.6305 - loss: 1.0583 - val_accuracy: 0.6214 - val_loss: 1.1050
Epoch 4/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 54s 92ms/step - accuracy: 0.6649 - loss: 0.9655 - val_accuracy: 0.6531 - val_loss: 1.0053
Epoch 5/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 53s 90ms/step - accuracy: 0.6940 - loss: 0.8814 - val_accuracy: 0.6662 - val_loss: 0.9591
Epoch 6/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 83s 92ms/step - accuracy: 0.7147 - loss: 0.8190 - val_accuracy: 0.6736 - val_loss: 0.9768
Epoch 7/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 80s 89ms/step - accuracy: 0.7401 - loss: 0.7554 - val_accuracy: 0.6745 - val_loss: 0.9422
Epoch 8/10
591/591 ━━━━━━━━━━━━━━━━━━━━ 54s 91ms/step - accuracy: 0.7581 - loss: 0.6986 - 